In [4]:

import os
import pandas as pd
import yfinance as yf
from fredapi import Fred
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Setup works!")


Setup works!


In [ ]:
# Compute the monthly returns, resample all prices to month-end


In [5]:
# 2. Download Market Data
tickers = ['SPY', 'GLD', 'TLT', 'XRE.TO', 'XIU.TO', 'TIPS', 'USO', 'VNQ', '^TNX']

data = yf.download(tickers, start='2010-01-01', end='2025-01-01', auto_adjust=True)

# Handle both single and multi-index structures
if isinstance(data.columns, pd.MultiIndex):
    prices = data['Close']
else:
    prices = data[['Close']] if 'Close' in data.columns else data

prices.columns = tickers[:len(prices.columns)]
prices.to_csv("data/raw/market_adjusted_prices.csv")

print("✅ Market data saved: data/raw/market_adj_prices.csv")
prices.tail()



[*********************100%***********************]  9 of 9 completed

✅ Market data saved: data/raw/market_adj_prices.csv


,SPY,GLD,TLT,XRE.TO,XIU.TO,TIPS,USO,VNQ,^TNX
Date,,,,,,,,,
2024-12-24,241.440002,596.076904,0.0270,84.754929,73.650002,86.810730,36.848515,14.306484,4.591
2024-12-26,243.070007,596.116638,0.0280,84.706703,73.129997,87.043747,NaN,NaN,4.579
2024-12-27,241.399994,589.841553,0.0200,84.012215,73.849998,86.169914,36.799541,14.268462,4.619
2024-12-30,240.630005,583.110535,0.0132,84.687408,74.820000,85.742706,36.535076,14.113987,4.545
2024-12-31,242.130005,580.989136,0.0190,84.234062,75.550003,86.490326,36.701595,14.296286,4.573


In [6]:
data = yf.download(tickers= tickers, start='2010-01-01', end='2025-01-01', auto_adjust=False)

if isinstance(data.columns, pd.MultiIndex):
    data = data.loc[:, ('Close', slice(None))]
    data.columns = data.columns.droplevel(0) # Drop the 'Close' level
else:
    data = data[['Close']]

data.to_csv("data/raw/market_prices.csv")
data.tail()

[*********************100%***********************]  9 of 9 completed


Ticker,GLD,SPY,TIPS,TLT,USO,VNQ,XIU.TO,XRE.TO,^TNX
Date,,,,,,,,,
2024-12-24,241.440002,601.299988,0.0270,87.870003,73.650002,89.410004,37.619999,15.05,4.591
2024-12-26,243.070007,601.340027,0.0280,87.820000,73.129997,89.650002,NaN,NaN,4.579
2024-12-27,241.399994,595.010010,0.0200,87.099998,73.849998,88.750000,37.570000,15.01,4.619
2024-12-30,240.630005,588.219971,0.0132,87.800003,74.820000,88.309998,37.299999,14.71,4.545
2024-12-31,242.130005,586.080017,0.0190,87.330002,75.550003,89.080002,37.470001,14.90,4.573


Gathered key assets
Extracted the closing prices for each asset
Created 2 files, market adjusted prices and market prices
Will be using the adjusted prices moving forward
adjusted prices accounts for prices changing due to stock splits, dividend payouts, rights offerings
Use for time series analysis since prices stay constant after events through time
Easier to compare with other assets or stocks for long term performance tracking

Next gather other useful data

Use the FRED (Federal Reserve Economic Data) API
download data about the economy
    CPI : Consumer Price Index
        use CPIAUCSL (consumer price index for all urban consumer, US)
        For inflation 
    10-year treasury yield
        measure interset rate trends
        inflation expectations
    gold prices 
        inflation hedge asset
    crude oil price 
        energy compnent of inflation
    read gdp
        tracks economic cycles
    unemployement rate
        labour market strength
    red funds rate
        policy response to inflation 

In [7]:
from fredapi import Fred
import pandas as pd

# Access the fred api 
fred = Fred(api_key = 'a308c955853d97775a6430c5ceb28c85')


cpi = fred.get_series('CPIAUCSL')
cpi = cpi.to_frame(name = 'CPI')
cpi.index.name = 'Date'
cpi.to_csv('data/raw/cpi_raw.csv')
print(" CPI data saved to data/raw/cpi_raw.csv")
cpi.tail()

 CPI data saved to data/raw/cpi_raw.csv


,CPI
Date,
2025-05-01,320.580
2025-06-01,321.500
2025-07-01,322.132
2025-08-01,323.364
2025-09-01,324.368


In [8]:
macro_codes = {
    'DGS10': '10Y_Treasury_Yield',
    'DCOILWTICO': 'Crude_Oil_Price',
    'UNRATE': 'Unemployment_Rate',
    'FEDFUNDS': 'Fed_Funds_Rate'
}

macro_df = pd.DataFrame()

for code, name in macro_codes.items():
    series = fred.get_series(code)
    if series.empty:
        print("⚠️ no data")
        continue
    series.name = name
    macro_df = pd.concat([macro_df, series], axis=1) # using concat doesn't get me all the dates, pandas still only take the union of all possible dates 
    print("✅ done")

macro_df.index = pd.to_datetime(macro_df.index)
macro_df.index.name = 'Date'

macro_df.tail()

✅ done
✅ done
✅ done
✅ done


,10Y_Treasury_Yield,Crude_Oil_Price,Unemployment_Rate,Fed_Funds_Rate
Date,,,,
2024-09-01,NaN,NaN,4.1,5.13
2024-12-01,NaN,NaN,4.1,4.48
2025-02-01,NaN,NaN,4.1,4.33
2025-03-01,NaN,NaN,4.2,4.33
2025-06-01,NaN,NaN,4.1,4.33


In [9]:
macro_codes = {
    'DGS10': '10Y_Treasury_Yield',
    'DCOILWTICO': 'Crude_Oil_Price',
    'UNRATE': 'Unemployment_Rate',
    'FEDFUNDS': 'Fed_Funds_Rate'
}


macro_df = pd.DataFrame()                       # start empty

for code, name in macro_codes.items():
    s = fred.get_series(code)
    if s.empty:
        print('⚠️ no data')
        continue
    macro_df[name] = s                          # <-- overwrite / create
    print('✅ done')

macro_df.index = pd.to_datetime(macro_df.index)
macro_df.index.name = 'Date'

macro_monthly = macro_df.resample("ME").last().dropna(how="all")
macro_monthly.tail()


macro_monthly.to_csv("data/raw/macro_monthly_data.csv")
print("macro_monthly saved to data/raw/macro_monthly_data.csv")

macro_df.to_csv("data/raw/macro_data.csv")
print("macro__df saved to data/raw/macro_data.csv")

✅ done
✅ done
✅ done
✅ done
macro_monthly saved to data/raw/macro_monthly_data.csv
macro__df saved to data/raw/macro_data.csv


Now that we have cpi and other marco data, combine all data into 1 data frame

merge and process


In [10]:
import pandas as pd
from pathlib import Path



# Load data
market = pd.read_csv("data/raw/market_adjusted_prices.csv", parse_dates=["Date"], index_col="Date")
macro = pd.read_csv("data/raw/macro_data.csv", parse_dates=["Date"], index_col="Date")
cpi = pd.read_csv("data/raw/cpi_raw.csv", parse_dates=["Date"], index_col="Date")

# Convert to monthly frequency (last value each month)
market_m = market.resample("ME").last()
macro_m = macro.resample("ME").last()
cpi_m = cpi.resample("ME").last()

# Merge all datasets
merged = market_m.join([macro_m, cpi_m], how="inner").dropna()

merged.to_csv("data/processed/merged_baseline.csv")
print("merged dataset saved to data/processed/merged_baseline.csv")

merged.tail()


merged dataset saved to data/processed/merged_baseline.csv


,SPY,GLD,TLT,XRE.TO,XIU.TO,TIPS,USO,VNQ,^TNX,10Y_Treasury_Yield,Crude_Oil_Price,Unemployment_Rate,Fed_Funds_Rate,CPI
Date,,,,,,,,,,,,,,
2024-05-31,215.300003,517.771790,0.0289,84.910545,74.820000,78.402512,32.423138,13.703003,4.514,4.51,77.97,4.0,5.33,313.140
2024-07-31,226.550003,542.529907,0.0390,89.591690,77.739998,86.214432,33.772900,14.815752,4.109,4.09,79.36,4.2,5.33,313.566
2024-08-31,231.289993,555.206360,0.0459,91.483154,74.339996,90.715385,34.299717,15.753405,3.911,3.91,74.52,4.2,5.33,314.131
2024-10-31,253.509995,561.809814,0.0477,88.222260,73.080002,90.536972,35.620800,15.323227,4.284,4.28,69.58,4.1,4.83,315.564
2024-11-30,245.589996,595.312500,0.0277,89.975410,71.610001,94.393082,37.984726,15.219061,4.178,4.18,68.26,4.2,4.64,316.449


compute monthly returns and basic stat

Create simply portfolio 

plot performance (cumulative returns) 

before adding more stats methods, 
    continue to mini 2 (build sql database)
        uploading raw files and cleaning + transforming with sql instead of python pandas 
        make data auto update and reproducable
        make sure github branches is correct
    continue to mini 2 (connect sql to powerbi) 
        build:
        Asset performance dashboard
        Rolling correlations
        Portfolio composition visuals
        Inflation vs asset returns chart